In [1]:
import pandas as pd
import numpy as np
from scipy.special import logsumexp
from scipy.optimize import minimize

In [2]:
# ingest data.csv

df = pd.read_csv('/Users/jamesb/Code/Pricing/Pricing_Project/data/data.csv', delimiter=',')

df.head(50)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/jamesb/Code/Pricing/Pricing_Project/data/data.csv'

In [ ]:


features = [
    'prop_starrating',
    'prop_review_score',
    'prop_brand_bool',
    'prop_location_score',
    'prop_accesibility_score',
    'prop_log_historical_price',
    'price_usd',
    'promotion_flag'
]

X = df[features].values
y = df['booking_bool'].values
groups = df['srch_id'].values

def compute_utilities(beta, X):
    return beta[0] + X @ beta[1:]

In [ ]:


def neg_log_likelihood(beta, X, y, groups):
    u = compute_utilities(beta, X)

    log_likelihood = 0.0
    for q in np.unique(groups):
        mask = groups == q
        u_q = u[mask]
        y_q = y[mask]

        log_likelihood += y_q @ u_q                       # utility of chosen hotel (0 if no purchase)
        log_likelihood -= logsumexp(np.append(u_q, 0.0))  # log(1 + sum exp(u_j)), 0 = no-purchase option

    return -log_likelihood

In [ ]:


beta_init = np.zeros(9)  # 1 intercept + 8 feature coefficients

result = minimize(
    neg_log_likelihood,
    beta_init,
    args=(X, y, groups),
    method='L-BFGS-B',
    options={'maxiter': 1000, 'disp': True}
)

beta_hat = result.x

param_names = ['intercept'] + features
for name, val in zip(param_names, beta_hat):
    print(f'{name:>30s}: {val:+.4f}')

/var/folders/1f/3bt7_4f57pd4c8qqp56r2wf00000gn/T/ipykernel_38682/927578697.py:5: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  result = minimize(


                     intercept: -2.8345
               prop_starrating: +0.4795
             prop_review_score: +0.1187
               prop_brand_bool: +0.2281
           prop_location_score: +0.0173
       prop_accesibility_score: +0.5678
     prop_log_historical_price: -0.0352
                     price_usd: -0.0073
                promotion_flag: +0.4558


In [ ]:
# problem two, search the nested subsets. Guaranteed to be nested

In [ ]:
def optimal_assortment(df_hotels, beta_hat, features):
    df = df_hotels.copy()

    # compute preference weights v_j = exp(u_j)
    u = compute_utilities(beta_hat, df[features].values)
    df['v'] = np.exp(u)

    # sort by price descending — revenue-ordered property
    df = df.sort_values('price_usd', ascending=False).reset_index(drop=True)

    num = 0.0  # sum of p_j * v_j for chosen hotels
    den = 1.0  # 1 + sum of v_j (the 1 is the no-purchase option)
    chosen_idx = []

    for i, row in df.iterrows():
        p = row['price_usd']
        v = row['v']
        R_current = num / den

        if p > R_current:
            num += p * v
            den += v
            chosen_idx.append(i)
        else:
            break  # all remaining hotels are cheaper — safe to stop

    S = df.loc[chosen_idx]
    R_opt = num / den
    return S, R_opt

In [ ]:
data_dir = '/Users/jamesb/Code/Pricing/Pricing_Project/data/'

for i in range(1, 5):
    df_i = pd.read_csv(f'{data_dir}data{i}.csv')
    S, R_opt = optimal_assortment(df_i, beta_hat, features)
    print(f'--- data{i}.csv ---')
    print(f'Optimal assortment size: {len(S)} hotels')
    print(f'Expected revenue: ${R_opt:.2f}')
    print(S[['price_usd', 'v']].to_string())
    print()

--- data1.csv ---
Optimal assortment size: 18 hotels
Expected revenue: $107.34
    price_usd         v
0         158  0.201074
1         156  0.247453
2         154  0.152886
3         150  0.139803
4         147  0.383515
5         147  0.383515
6         145  0.166154
7         144  0.167375
8         140  0.172350
9         131  0.166335
10        131  0.163479
11        125  0.305364
12        121  0.128722
13        121  0.156204
14        120  0.199533
15        118  0.209612
16        112  0.353772
17        108  0.097011

--- data2.csv ---
Optimal assortment size: 10 hotels
Expected revenue: $130.99
   price_usd         v
0        233  0.290876
1        212  0.535090
2        206  0.354464
3        161  0.305108
4        147  0.434631
5        146  0.340530
6        143  0.348093
7        137  0.587492
8        134  0.414574
9        133  0.604954

--- data3.csv ---
Optimal assortment size: 18 hotels
Expected revenue: $121.04
    price_usd         v
0         603  0.003424
1   